# Evo 2 Geometric Stability Scaling Experiment

Tests the hypothesis: **Larger context window = different geometric stability** across the Evo 2 7B family.

## Architecture

Evo 2 uses a **StripedHyena** backbone — alternating Hyena long-convolution layers and attention layers. All three 7B checkpoints run locally on an A100 (bf16, no FP8 required).

## Checkpoints (7B family)

| Checkpoint     | Parameters | Context | Hardware          |
|----------------|------------|---------|-------------------|
| `evo2_7b_base` | 7B         | 8K      | A100 40GB (bf16)  |
| `evo2_7b_262k` | 7B         | 262K    | A100 40GB (bf16)  |
| `evo2_7b`      | 7B         | 1M      | A100 40GB (bf16)  |

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py` to `/content/`
2. Change Runtime to **GPU** (A100 recommended)
3. Run cells in order

---

In [ ]:
# Install Dependencies
#
# All 7B checkpoints run locally via the evo2 (Vortex) package.
# No API key or Transformer Engine required.

print('Installing core dependencies...')
!pip install -q shesha-geometry matplotlib seaborn pandas scipy
!pip install -q ninja
import sys, os

# Pin torch to 2.7.1+cu128 (Arc Institute recommended for evo2/flash-attn)
print('Pinning torch to 2.7.1+cu128...')
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128

import torch
print(f'torch {torch.__version__} | CUDA {torch.version.cuda}')

# Build flash-attn from source (~10-15 min on A100, one-time cost)
print('Building flash-attn 2.8.0.post2 from source (~10-15 min)...')
!pip install flash-attn==2.8.0.post2 --no-build-isolation

!pip install -q evo2

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from evo2 import Evo2
print('\nEvo2 package loaded successfully')
print('Done!')

In [ ]:
# Configuration

PHASE = 'full'  # 'quick' or 'full'

import os, sys
sys.path.insert(0, '.')

OUTPUT_BASE = './results/evo2_scaling_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# 7B checkpoints only -- all run locally on A100 (bf16, no FP8 needed)
# (name, param_B, context_tokens, label)
ALL_CHECKPOINTS = [
    ('evo2_7b_base', 7.0,  8192,    'Evo2-7B-8K'),
    ('evo2_7b_262k', 7.0,  262144,  'Evo2-7B-262K'),
    ('evo2_7b',      7.0,  1048576, 'Evo2-7B-1M'),
]

# Embedding layer at ~75% model depth (per Evo 2 paper; 7B has 32 blocks -> block 28)
EMBEDDING_LAYERS = {
    'evo2_7b_base': 'blocks.28.mlp.l3',
    'evo2_7b_262k': 'blocks.28.mlp.l3',
    'evo2_7b':      'blocks.28.mlp.l3',
}

BATCH_OVERRIDES = {
    'evo2_7b_base': 4,
    'evo2_7b_262k': 4,
    'evo2_7b':      2,
}

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 1000,
        'batch_size': 4,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 1000,
        'batch_size': 4,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

active_checkpoints = ALL_CHECKPOINTS if torch.cuda.is_available() else []
if not torch.cuda.is_available():
    print('WARNING: no GPU detected — no checkpoints will run')

config = CONFIG[PHASE]
print(f'Phase: {PHASE.upper()}')
print(f'Architecture: Evo 2 (StripedHyena 2) — 7B family')
print(f'Sequences: {config["n_sequences"]}  |  Length: {config["seq_length"]} nt')
print(f'Active checkpoints: {len(active_checkpoints)}')
for ckpt_name, pb, cl, lbl in active_checkpoints:
    print(f'  {lbl}  ({pb:.0f}B, {cl:,} ctx)')
print(f'Results: {RESULTS_DIR}')

In [ ]:
# Generate Synthetic DNA Sequences & Perturbation Suite

import numpy as np
from dataclasses import dataclass, field

DNA_BASES = ['A', 'C', 'G', 'T']
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}


def generate_dna_sequences(n_sequences, seq_length, seed=320):
    """Generate random DNA sequences of exact length."""
    rng = np.random.default_rng(seed)
    sequences = [
        ''.join(rng.choice(DNA_BASES, size=seq_length))
        for _ in range(n_sequences)
    ]
    print(f'Generated {len(sequences)} DNA sequences (length={seq_length})')
    print(f'Sample: {sequences[0][:50]}...')
    return sequences


def mutate_dna(sequence, mutation_rate, rng):
    """SNP mutations at the given rate."""
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))


@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f'snp_{int(rate * 100)}pct'
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(
                name='reverse_complement', category='rc',
                sequences=[reverse_complement(s) for s in sequences],
                params={}, description='Reverse complement transformation',
            )
        return results


sequences = generate_dna_sequences(config['n_sequences'], config['seq_length'])

suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])

# Quick validation
test_perturbed = suite.run_all(sequences[:5])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:5], pset.sequences)
    ]
    print(f'  {name}: mean_hamming={np.mean(dists):.4f}')
print('Perturbation suite ready')

In [ ]:
# Evo 2 Local Embedding Wrapper (7B via Vortex)

import torch
import gc
import numpy as np
from evo2 import Evo2


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f'  GPU memory after cleanup: {mem:.2f} GB')


def make_evo2_embedding_fn(checkpoint_name, batch_size=4):
    """Load an Evo 2 checkpoint via Vortex and return an embedding function."""
    print(f'Loading {checkpoint_name}...')
    device = 'cuda:0'
    effective_batch = BATCH_OVERRIDES.get(checkpoint_name, batch_size)

    evo2_model = Evo2(checkpoint_name)
    n_params = sum(p.numel() for p in evo2_model.model.parameters()) / 1e6
    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f'  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB')

    layer_name = EMBEDDING_LAYERS[checkpoint_name]
    print(f'  Embedding layer: {layer_name}')

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_total = len(sequences)
        for idx, seq in enumerate(sequences):
            if (idx + 1) % 50 == 0 or idx == 0 or idx == n_total - 1:
                print(f'    Seq {idx+1}/{n_total}', end='\r')
            input_ids = torch.tensor(
                evo2_model.tokenizer.tokenize(seq), dtype=torch.int,
            ).unsqueeze(0).to(device)
            _, layer_embs = evo2_model(
                input_ids, return_embeddings=True, layer_names=[layer_name],
            )
            emb = layer_embs[layer_name]
            embeddings.append(emb.mean(dim=1).squeeze(0).cpu().float().numpy())
        print()
        return np.array(embeddings, dtype=np.float32)

    return embed, evo2_model, n_params


print('Evo 2 local (Vortex) wrapper ready — 7B checkpoints on A100')

In [ ]:
# Evaluation Harness (shesha-geometry)

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f'Shesha StabilityHarness configured (bootstrap={config["n_bootstrap"]})')

In [ ]:
# Run Experiment

import time
import pandas as pd
from evaluation_harness import ModelReport

print('=' * 70)
print(f'EVO 2 (STRIPEDHYENA 2) SCALING -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []

for ckpt_idx, (ckpt_name, params_b, ctx_len, label) in enumerate(active_checkpoints):
    print(f"\n{'='*70}")
    print(f'[{ckpt_idx+1}/{len(active_checkpoints)}] {label} ({ckpt_name})')
    print(f'  Params: {params_b:.0f}B | Context: {ctx_len:,}')
    print('=' * 70)

    start_time = time.time()

    embed_fn, evo2_model_obj, n_params_m = make_evo2_embedding_fn(
        ckpt_name, config['batch_size']
    )
    model_param_counts.append(n_params_m)

    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = ckpt_name.replace('/', '_').replace('-', '_')
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    # Clean embeddings
    if os.path.exists(cache_clean):
        print(f'Loading cached clean embeddings: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    # Perturbed embeddings
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    # Free GPU memory (only for local models)
    if evo2_model_obj is not None:
        del evo2_model_obj
        cleanup_gpu()

    # Run Shesha evaluation
    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=label,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'checkpoint': ckpt_name,
            'label': label,
            'params_B': params_b,
            'size_M': round(n_params_m, 1),
            'context_length': ctx_len,
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted {label} in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Stability:     {summary["mean_perturbation_stability_score"]:.4f}')

# Save detailed results
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/evo2_scaling_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Visualization -- Context Scaling (8K / 262K / 1M)

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12})

# --- Panel 1: Parameter scaling at 8K context ---
df_8k = df_detailed[df_detailed['context_length'] == 8192].copy()
df_1m = df_detailed[df_detailed['context_length'] == 1048576].copy()

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

for col_idx, (metric_col, ylabel, color, title) in enumerate([
    ('composite', 'Composite Stability', '#2563EB', 'A. Composite Stability'),
    ('rdm_similarity', 'RDM Similarity', '#16A34A', 'B. RDM Similarity'),
    ('pert_magnitude', 'Perturbation Magnitude', '#DC2626', 'C. Perturbation Magnitude'),
]):
    ax = axes[col_idx]

    # 8K context models (parameter scaling)
    if len(df_8k) > 0:
        agg = df_8k.groupby('params_B').agg(
            mean=(metric_col, 'mean'),
            std=(metric_col, 'std'),
        ).reset_index().sort_values('params_B')
        ax.errorbar(agg['params_B'], agg['mean'], yerr=agg['std'],
                    fmt='o-', color=color, linewidth=2.5, markersize=10,
                    capsize=5, label='8K context (base)')

    # 1M context models (parameter scaling)
    if len(df_1m) > 0:
        agg2 = df_1m.groupby('params_B').agg(
            mean=(metric_col, 'mean'),
            std=(metric_col, 'std'),
        ).reset_index().sort_values('params_B')
        ax.errorbar(agg2['params_B'], agg2['mean'], yerr=agg2['std'],
                    fmt='s--', color=color, linewidth=2, markersize=9,
                    capsize=5, alpha=0.7, label='1M context')

    ax.set_xscale('log')
    ax.set_xlabel('Parameters (B)')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=9)

fig.suptitle('Evo 2: Geometric Stability vs. Parameter Count',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evo2_param_scaling_{PHASE}.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved to {RESULTS_DIR}/evo2_param_scaling_{PHASE}.png')

In [ ]:
# Visualization -- Context-Window Scaling (fixed params = 7B)
#
# Compare 7B-8K vs 7B-262K vs 7B-1M to isolate context-window scaling
# at fixed parameter count.

df_7b = df_detailed[df_detailed['params_B'] == 7.0].copy()

if len(df_7b) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

    for col_idx, (metric_col, ylabel, color, title) in enumerate([
        ('composite', 'Composite Stability', '#7C3AED', 'A. Composite Stability'),
        ('rdm_similarity', 'RDM Similarity', '#0891B2', 'B. RDM Similarity'),
        ('pert_magnitude', 'Perturbation Magnitude', '#EA580C', 'C. Perturbation Magnitude'),
    ]):
        ax = axes[col_idx]
        agg = df_7b.groupby('context_length').agg(
            mean=(metric_col, 'mean'),
            std=(metric_col, 'std'),
        ).reset_index().sort_values('context_length')
        ax.errorbar(agg['context_length'], agg['mean'], yerr=agg['std'],
                    fmt='D-', color=color, linewidth=2.5, markersize=10, capsize=5)
        ax.set_xscale('log')
        ax.set_xlabel('Context Window (tokens)')
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold')
        ax.grid(True, alpha=0.2)

    fig.suptitle('Evo 2 (7B): Geometric Stability vs. Context Window',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/evo2_context_scaling_{PHASE}.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved to {RESULTS_DIR}/evo2_context_scaling_{PHASE}.png')
else:
    print('No 7B checkpoints available -- skipping context-window plot')

In [ ]:
# Per-Perturbation Breakdown Heatmap

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(active_checkpoints))))

# Heatmap: composite stability
pivot_comp = df_detailed.pivot_table(
    values='composite', index='perturbation', columns='label',
)
# Reorder columns by params then context
col_order = [lbl for _, _, _, lbl in active_checkpoints]
pivot_comp = pivot_comp[[c for c in col_order if c in pivot_comp.columns]]

ax = axes[0]
sns.heatmap(pivot_comp, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Composite'})
ax.set_title('Composite Stability by Perturbation', fontweight='bold')
ax.set_xlabel('Evo 2 Checkpoint')
ax.set_ylabel('Perturbation')

# Heatmap: RDM similarity
pivot_rdm = df_detailed.pivot_table(
    values='rdm_similarity', index='perturbation', columns='label',
)
pivot_rdm = pivot_rdm[[c for c in col_order if c in pivot_rdm.columns]]

ax = axes[1]
sns.heatmap(pivot_rdm, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'RDM Similarity'})
ax.set_title('RDM Similarity by Perturbation', fontweight='bold')
ax.set_xlabel('Evo 2 Checkpoint')
ax.set_ylabel('Perturbation')

fig.suptitle('Evo 2: Per-Perturbation Breakdown (All Checkpoints)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evo2_scaling_{PHASE}_heatmap.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved to {RESULTS_DIR}/evo2_scaling_{PHASE}_heatmap.png')

In [ ]:
# Cross-Architecture Comparison
#
# Overlay Evo 2 with results from other architectures.

import glob

fig, ax = plt.subplots(figsize=(11, 7))

# Evo 2 -- 8K context (parameter scaling axis)
if len(df_8k) > 0:
    evo_8k_agg = df_8k.groupby(['label', 'size_M'])['composite'].mean().reset_index()
    evo_8k_agg = evo_8k_agg.sort_values('size_M')
    ax.plot(evo_8k_agg['size_M'], evo_8k_agg['composite'], 'p-',
            color='#7C3AED', linewidth=2.5, markersize=12,
            label='Evo 2 (StripedHyena, 8K ctx)', zorder=5)

# Evo 2 -- 1M context
if len(df_1m) > 0:
    evo_1m_agg = df_1m.groupby(['label', 'size_M'])['composite'].mean().reset_index()
    evo_1m_agg = evo_1m_agg.sort_values('size_M')
    ax.plot(evo_1m_agg['size_M'], evo_1m_agg['composite'], 'p--',
            color='#7C3AED', linewidth=2, markersize=10, alpha=0.7,
            label='Evo 2 (StripedHyena, 1M ctx)', zorder=4)

# Other architectures
base_search = f'./results'
for arch_label, pattern, color, marker in [
    ('NT (Transformer)',   '**/nt_scaling_*_detailed.csv',       'tab:orange', 's-'),
    ('ESM-2 (Transformer)','**/esm2_scaling_*_detailed.csv',     'tab:red',    'o-'),
    ('Caduceus (SSM)',     '**/caduceus_scaling_*_detailed.csv',  'tab:blue',   'D-'),
    ('GPN (Conv)',         '**/gpn_scaling_*_detailed.csv',       'tab:cyan',   '^-'),
    ('HyenaDNA (Long Conv)', '**/hyenadna_scaling_*_detailed.csv', 'tab:olive', 'v-'),
    ('DNABERT-2',          '**/dnabert2_scaling_*_detailed.csv',  'tab:green',  'P'),
]:
    files = glob.glob(f'{base_search}/{pattern}', recursive=True)
    if files:
        df_other = pd.read_csv(files[0])
        agg = df_other.groupby('size_M')['composite'].mean().reset_index()
        ax.plot(agg['size_M'], agg['composite'], marker,
                color=color, linewidth=2, markersize=9, label=arch_label)

ax.set_xscale('log')
ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('Full Architecture Comparison: The Geometric Tax\n'
             '(Evo 2 = StripedHyena Multi-Hybrid)',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=9, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evo2_crossexp_{PHASE}.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Slope analysis
print('\n--- Slope Analysis ---')
if len(df_8k) > 0:
    evo_8k_slope = evo_8k_agg['composite'].iloc[-1] - evo_8k_agg['composite'].iloc[0]
    print(f'Evo 2 (8K) composite change: {evo_8k_slope:+.4f} (smallest to largest)')
if len(df_1m) > 0 and len(evo_1m_agg) >= 2:
    evo_1m_slope = evo_1m_agg['composite'].iloc[-1] - evo_1m_agg['composite'].iloc[0]
    print(f'Evo 2 (1M) composite change: {evo_1m_slope:+.4f}')
if len(df_7b) > 0:
    df_7b_agg = df_7b.groupby('context_length')['composite'].mean().reset_index().sort_values('context_length')
    ctx_slope = df_7b_agg['composite'].iloc[-1] - df_7b_agg['composite'].iloc[0]
    print(f'Evo 2 7B context-window change: {ctx_slope:+.4f} (8K to 1M)')

print(f'\nSaved to {RESULTS_DIR}/evo2_crossexp_{PHASE}.png')